# English → Dutch Transformer — Full Pipeline (Train + Inference + Attention + Beam Search)
One notebook from original `hkproj/pytorch-transformer` (`Colab_Train.ipynb` + `Inference.ipynb` + `attention_visual.ipynb` + `Beam_Search.ipynb` merged). Your `pytorch-transformer/` already set to `en-nl` (Dutch = `nl`, full name Dutch) in `config.py:11` — 38,652 ex, ~3.5h/20ep on T4 like Italian.
Order: Setup → Config → Train → Inference (greedy) → Validation → Attention Visual → Beam Search. You can run all, or skip Train if weights already on Drive.

## 0. Setup — GPU & Dependencies

In [ ]:
import torch, warnings; warnings.filterwarnings('ignore')
print(f"torch {torch.__version__} | cuda={torch.cuda.is_available()} | device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")
!nvidia-smi -L

In [ ]:
%%capture
!pip install -q datasets tokenizers torchmetrics tensorboard altair pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/neural-network-projects/pytorch-transformer_weights
!mkdir -p /content/drive/MyDrive/neural-network-projects/pytorch-transformer_runs

## 1. Get Code & Config (en → nl Dutch)

In [ ]:
import pathlib, os
if not pathlib.Path("/content/neural-network-projects").exists():
    !git clone https://github.com/abhijitdalal26/neural-network-projects.git /content/neural-network-projects
%cd /content/neural-network-projects/pytorch-transformer
!ls -lh model.py dataset.py config.py train.py translate.py
!cat config.py

In [ ]:
from config import get_config, get_weights_file_path, latest_weights_file_path
cfg = get_config()
print("Original cfg:", cfg)
assert cfg['lang_src']=='en' and cfg['lang_tgt']=='nl', "should be en-nl (Dutch)"
# Persist to Drive so you don't retrain each time — this is what 'if model already available' means
cfg['model_folder'] = "/content/drive/MyDrive/neural-network-projects/pytorch-transformer_weights"
cfg['tokenizer_file'] = "/content/drive/MyDrive/neural-network-projects/pytorch-transformer_weights/tokenizer_{0}.json"
cfg['experiment_name'] = "/content/drive/MyDrive/neural-network-projects/pytorch-transformer_runs/tmodel_en_nl"
cfg['num_epochs'] = 20  # set 3-5 for quick smoke test
cfg['batch_size'] = 8
cfg['preload'] = 'latest'  # if Drive already has tmodel_*.pt, resume from there
print("Effective cfg:", cfg)
print("Latest checkpoint on Drive:", latest_weights_file_path(cfg))
print("\n'model already available' = latest_weights_file_path(cfg) != None → train_model will set initial_epoch = saved_epoch+1 and skip 0..saved. If you open notebook just to demo, we can SKIP the 3.5h train cell entirely.")

## 2. Train — `train.py:184` (uses `model.py:211` + `dataset.py:5`)
Does `get_ds:141` (opus_books en-nl 38k train/val split), `get_model:174`, loop `encode:238 / decode:239 / project:240 loss.backward:238` per batch, `run_validation:57` BLEU/CER each epoch, saves `tmodel_XX.pt:262` to Drive. Skip if you already have weights (see `SKIP_TRAIN`).

In [ ]:
SKIP_TRAIN = False  # set True to skip 3.5h and jump to Inference/Visuals if weights exist
import pathlib as _pl
ckpt = latest_weights_file_path(cfg)
if SKIP_TRAIN and ckpt is not None:
    print(f"Skipping train — using existing {ckpt}")
else:
    # Also check manual: if latest already at epoch 19 (full 20ep), no need to retrain
    if ckpt is not None and "tmodel_19" in ckpt and not SKIP_TRAIN:
        print(f"Found full training {ckpt} — set SKIP_TRAIN=True next time to save 3.5h")
    from train import train_model
    train_model(cfg)  # ~0.6h/epoch en-nl → ~3.5h/20ep T4; interruption resumes via preload=latest

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/neural-network-projects/pytorch-transformer_runs --port 6006

## 3. Inference — Greedy Decode `train.py:25` / `translate.py:10` (<1s/sentence)
Loads tokenizer Drive json + `latest_weights` + `model.encode/decode/project` loop until `[EOS]`.

In [ ]:
from translate import translate
import config as cfgmod
orig = cfgmod.get_config; cfgmod.get_config = lambda: cfg  # make translate use Drive cfg
print(translate("Hello, how are you?"))
print("---")
print(translate("I love learning languages."))
print("---")
print(translate("The book is on the table."))
cfgmod.get_config = orig
print("\n# By index (shows SOURCE/TARGET/PREDICTED from val set):")
cfgmod.get_config = lambda: cfg; print(translate(42)); cfgmod.get_config = orig

## 4. Validation — BLEU/WER/CER `train.py:56`

In [ ]:
import torch
from train import get_ds, get_model, run_validation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device", device)
train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt = get_ds(cfg)
model = get_model(cfg, tokenizer_src.get_vocab_size(), tokenizer_tgt.get_vocab_size()).to(device)
ckpt = latest_weights_file_path(cfg)
print("ckpt", ckpt)
state = torch.load(ckpt, map_location=device)
model.load_state_dict(state['model_state_dict'])
print(f"Loaded epoch {state['epoch']} global_step {state['global_step']}")
run_validation(model, val_dataloader, tokenizer_src, tokenizer_tgt, cfg['seq_len'], device, lambda m: print(m), state['global_step'], None, num_examples=5)

## 5. Attention Visual — `attention_visual.ipynb` + `model.py:135` 
Heatmaps of `attention_scores` per `Encoder(6 layers) / Decoder(6)`. Needs trained ckpt from §2. Shows `model.encoder.layers[l].self_attention_block.attention_scores[0, head]` etc.

In [ ]:
import altair as alt, pandas as pd, numpy as np
from train import greedy_decode
from config import get_weights_file_path
# reuse model/tokenizers from §4 (already loaded) — ensure attn scores populated by running greedy once
def load_next_batch():
    batch = next(iter(val_dataloader))
    encoder_input = batch["encoder_input"].to(device)
    encoder_mask = batch["encoder_mask"].to(device)
    decoder_input = batch["decoder_input"].to(device)
    decoder_mask = batch["decoder_mask"].to(device)
    enc_tokens = [tokenizer_src.id_to_token(idx) for idx in encoder_input[0].cpu().numpy()]
    dec_tokens = [tokenizer_tgt.id_to_token(idx) for idx in decoder_input[0].cpu().numpy()]
    assert encoder_input.size(0)==1
    model_out = greedy_decode(model, encoder_input, encoder_mask, tokenizer_src, tokenizer_tgt, cfg['seq_len'], device)
    return batch, enc_tokens, dec_tokens

def mtx2df(m, max_row, max_col, row_tokens, col_tokens):
    return pd.DataFrame([ (r,c,float(m[r,c]), f"{r:03d} {row_tokens[r] if len(row_tokens)>r else '<blank>'}", f"{c:03d} {col_tokens[c] if len(col_tokens)>c else '<blank>'}") for r in range(m.shape[0]) for c in range(m.shape[1]) if r<max_row and c<max_col], columns=["row","column","value","row_token","col_token"])
def get_attn_map(attn_type: str, layer: int, head: int):
    if attn_type=="encoder": attn = model.encoder.layers[layer].self_attention_block.attention_scores
    elif attn_type=="decoder": attn = model.decoder.layers[layer].self_attention_block.attention_scores
    else: attn = model.decoder.layers[layer].cross_attention_block.attention_scores
    return attn[0, head].data
def attn_map(attn_type, layer, head, row_tokens, col_tokens, max_len):
    df = mtx2df(get_attn_map(attn_type, layer, head), max_len, max_len, row_tokens, col_tokens)
    return alt.Chart(df).mark_rect().encode(x=alt.X("col_token", axis=alt.Axis(title="")), y=alt.Y("row_token", axis=alt.Axis(title="")), color="value", tooltip=["row","column","value","row_token","col_token"]).properties(height=400,width=400,title=f"{attn_type} L{layer} H{head}").interactive()
def get_all_attention_maps(attn_type, layers, heads, row_tokens, col_tokens, max_len):
    charts=[]
    for layer in layers:
        rowCharts=[attn_map(attn_type, layer, head, row_tokens, col_tokens, max_len) for head in heads]
        charts.append(alt.hconcat(*rowCharts))
    return alt.vconcat(*charts)

In [ ]:
batch, enc_tokens, dec_tokens = load_next_batch()
print(f"SOURCE: {batch['src_text'][0]}")
print(f"TARGET: {batch['tgt_text'][0]}")
sent_len = enc_tokens.index("[PAD]") if "[PAD]" in enc_tokens else 20
print(f"sent_len {sent_len} trunc 20")
layers=[0,1,2]; heads=[0,1,2,3,4,5,6,7]
get_all_attention_maps("encoder", layers, heads, enc_tokens, enc_tokens, min(20, sent_len))

In [ ]:
get_all_attention_maps("decoder", [0,1,2], [0,1,2,3,4,5,6,7], dec_tokens, dec_tokens, min(20, sent_len))

In [ ]:
get_all_attention_maps("encoder-decoder", [0,1,2], [0,1,2,3,4,5,6,7], enc_tokens, dec_tokens, min(20, sent_len))

## 6. Beam Search — `Beam_Search.ipynb` (quality vs speed)
Greedy (`beam=1` `train.py:25`) picks `argmax` each step; Beam keeps top-k candidates (`beam_size=4`) by summed `log_prob`, better BLEU but 4× slower. Uses `causal_mask` `dataset.py:88` each expansion.

In [ ]:
from dataset import causal_mask
def beam_search_decode(model, beam_size, source, source_mask, tokenizer_src, tokenizer_tgt, max_len, device):
    sos_idx = tokenizer_tgt.token_to_id('[SOS]'); eos_idx = tokenizer_tgt.token_to_id('[EOS]')
    encoder_output = model.encode(source, source_mask)
    decoder_initial = torch.empty(1,1).fill_(sos_idx).type_as(source).to(device)
    candidates = [(decoder_initial, 0.0)]  # (seq, log_score)
    while True:
        if any(c.size(1)==max_len for c,_ in candidates): break
        new_cands=[]
        for cand, score in candidates:
            if cand[0][-1].item()==eos_idx: new_cands.append((cand, score)); continue
            cand_mask = causal_mask(cand.size(1)).type_as(source_mask).to(device)
            out = model.decode(encoder_output, source_mask, cand, cand_mask)
            prob = model.project(out[:,-1])  # (1, vocab)
            log_prob = torch.log_softmax(prob, dim=1)
            topk_log, topk_idx = torch.topk(log_prob, beam_size, dim=1)
            for i in range(beam_size):
                token = topk_idx[0][i].unsqueeze(0).unsqueeze(0)
                new_cand = torch.cat([cand, token], dim=1)
                new_score = score + topk_log[0][i].item()
                new_cands.append((new_cand, new_score))
        # keep top beam_size
        candidates = sorted(new_cands, key=lambda x: x[1], reverse=True)[:beam_size]
        if all(c[0][-1].item()==eos_idx for c,_ in candidates): break
    return sorted(candidates, key=lambda x: x[1], reverse=True)[0][0].squeeze(0)  # best

# Demo on same batch as §5
batch = next(iter(val_dataloader))
src = batch["encoder_input"].to(device); src_mask=batch["encoder_mask"].to(device)
print("SOURCE:", batch["src_text"][0])
print("TARGET:", batch["tgt_text"][0])
greedy = greedy_decode(model, src, src_mask, tokenizer_src, tokenizer_tgt, cfg['seq_len'], device)
print("GREEDY  :", tokenizer_tgt.decode(greedy.cpu().numpy()))
beam4 = beam_search_decode(model, 4, src, src_mask, tokenizer_src, tokenizer_tgt, cfg['seq_len'], device)
print("BEAM(4) :", tokenizer_tgt.decode(beam4.cpu().numpy()))
beam8 = beam_search_decode(model, 8, src, src_mask, tokenizer_src, tokenizer_tgt, cfg['seq_len'], device)
print("BEAM(8) :", tokenizer_tgt.decode(beam8.cpu().numpy()))

### What 'model already available' means
- `latest_weights_file_path(cfg)` `config.py:26` glob `opus_books_weights/tmodel_*.pt`. If `tmodel_04.pt` exists, `train.py:212` `preload=latest` does `state = torch.load(ckpt); model.load_state_dict; initial_epoch = state['epoch']+1; optimizer.load; global_step=...` — so §2 continues from epoch 5 not 0.
- **For demo:** set `SKIP_TRAIN=True` in §2 cell if ckpt already at `tmodel_19.pt` — you jump straight to §3 Inference / §5 Visuals without 3.5h wait. Colab with Drive mount persists, local runtime discards — that's why we moved `model_folder/tokenizer_file` to `/content/drive/...`.